In [ ]:
# Полное обновление: тесты, таблица, история, карта, радар, README.
# Комментарий — отдельной строкой: всё, что после %run, IPython передаёт скрипту как аргументы.
%run ../scripts/update_progress.py

# Панель прогресса ROADMAP-ML

Мой подробный пульт (README — витрина для других, а здесь всё для меня). **Run All** — первая ячейка пересобирает всё (таблицу, карту, радар, README), дальше видно, где я сейчас, что изменилось и что сломано.

**Где что лежит**

| Что | Где | Кто меняет |
|---|---|---|
| Эталоны методов | `src/<этап>/<подраздел>/<метод>.py` или `.../<метод>/<метод>.py` | я: код + шапка `# status:` |
| Попытки (повторы) | `attempts/<slug>_attempt_NN.py` в папке метода, подраздела или этапа | я; засчитываются, только если прошли тест |
| Тесты | `tests/<этап>/test_<slug>.py` со строкой `IMPL = ("src/...py", "имя")` | я; гоняют и эталон, и все попытки |
| Таблица тем | `roadmap_tracker.csv` | скрипт пересобирает из `src/` и тестов |
| История | `progress_history.csv` | скрипт дописывает строку, когда проценты изменились |
| Картинки | `assets/progress.png`, `assets/radar.png` | скрипт |
| Витрина | `README.md` | скрипт (руками не править) |

**Что значат цифры**

- `status` в шапке: `todo` — не начинал, `in_progress` — в работе (оранжевая рамка), `ready` — закрыл.
- `learned` — из статуса: todo → **0**, in_progress → **0.5**, ready → **1**. Готовность этапа = среднее `learned`.
- `repeated` — сколько попыток прошло тест; `last_repeat` — дата последней по git; `score` — `repeated` минус 1 за каждый полный месяц простоя (0…10). **Цвет кубика = score / 10.**
- Штриховка — сколько **обязательных для этапа** задач не сделано: model — learned, readme, experiment, test; eda/preprocessing — learned, readme, experiment; остальные — learned, readme, test.
- `!` — тест эталона упал; `?` — нет статуса в шапке.

## 1. Что изменилось с последнего коммита

Сравниваю свежую таблицу с `roadmap_tracker.csv` из последнего коммита git.

In [ ]:
import subprocess
from io import StringIO
import pandas as pd
from IPython.display import Image, Markdown, display
pd.set_option("display.max_rows", 300, "display.max_colwidth", 80)

df, probs = TABLE, PROBLEMS                             # их оставил %run выше
committed = subprocess.run(["git", "show", "HEAD:roadmap_tracker.csv"], cwd=ROOT, capture_output=True, text=True).stdout
before = pd.read_csv(StringIO(committed.lstrip("\ufeff")), dtype=str, keep_default_na=False) if committed else pd.DataFrame(columns=COLUMNS)
after = df.astype(str)

b = before.set_index("path") if "path" in before.columns else pd.DataFrame()
a = after.set_index("path")
added, removed = sorted(set(a.index) - set(b.index)), sorted(set(b.index) - set(a.index))
changes = [{"path": p, "поле": c, "было": b.at[p, c], "стало": a.at[p, c]}
           for p in a.index.intersection(b.index) for c in a.columns
           if c in b.columns and b.at[p, c] != a.at[p, c]]
print(f"Тем: {len(after)} (в коммите {len(before)}), новых: {len(added)}, пропало: {len(removed)}, изменённых ячеек: {len(changes)}")
for p in added:
    print("  + ", p)
for p in removed:
    print("  − ", p)
display(pd.DataFrame(changes) if changes else Markdown("_С последнего коммита ничего не изменилось._"))

## 2. Проблемы

Упавшие тесты (эталоны и попытки) и файлы без статуса в шапке.

In [ ]:
lines = []
if probs["failed_tests"]:
    lines.append(f"**Упавшие тесты — {len(probs['failed_tests'])}**")
    lines += [f"- `{test}` — `{file}`" for test, file in probs["failed_tests"]]
if probs["no_status"]:
    lines.append(f"**Нет или неверный статус в шапке — {len(probs['no_status'])}**")
    lines += [f"- `{path}`" for path in probs["no_status"]]
display(Markdown("\n".join(lines) if lines else "**Проблем нет.**"))

## 3. Карта и радар

In [ ]:
display(Image(filename=str(ROOT / "assets" / "progress.png")))
display(Image(filename=str(ROOT / "assets" / "radar.png"), width=640))

## 4. Проценты по этапам и история

**Готовность** — среднее `learned`. **Повторения** — средний `score` / 10, как цвет кубиков. История пишется, только когда проценты меняются.

In [ ]:
display(stage_summary(df))
hist_path = ROOT / "progress_history.csv"
if hist_path.exists():
    hist = pd.read_csv(hist_path, encoding="utf-8-sig", parse_dates=["datetime"])
    display(hist.tail(10))
    if len(hist) > 1:
        hist.set_index("datetime")[HISTORY_COLS[1:]].plot(figsize=(9, 3.5), marker="o", ylabel="готовность, %", grid=True)

## 5. Повторы

Темы с засчитанными попытками: сколько, когда последняя (по git) и итог со штрафом за простой. Если `score` меньше `repeated` — пора повторить.

In [ ]:
d = numeric(df)
rep = d[d.repeated > 0][["slug", "stage", "repeated", "last_repeat", "score"]].sort_values("score")
display(rep if len(rep) else Markdown("_Засчитанных повторов пока нет — все кубики серые._"))

## 6. Изучено / в работе / в планах

In [ ]:
cols = ["slug", "stage", "subsection", "path", "description"]
for title, mask in [("Изучено (ready)", d.learned == 1), ("В работе (in_progress)", d.status == "in_progress")]:
    g = d[mask]
    display(Markdown(f"**{title} — {len(g)}**"))
    display(g[cols] if len(g) else Markdown("_пусто_"))

plans = d[d.status == "todo"]
display(Markdown(f"**В планах (todo) — {len(plans)}**, по этапам:"))
display(plans.groupby("stage", sort=False).size().rename("тем").to_frame())

In [ ]:
# Полный лог последнего запуска (logs/progress.log)
print((ROOT / "logs" / "progress.log").read_text(encoding="utf-8"))

## 7. Вся таблица

In [ ]:
df